# EP2 - IT Assist Multi-Agent

## Memoria de corto y largo plazo

Este notebook compara un agente sin memoria con otro que conserva contexto entre interacciones.

### Objetivo
Demostrar continuidad conversacional y recuperación de contexto en consultas relacionadas.

### Indicadores demostrados
- **IE3:** memoria de contenido.
- **IE4:** contexto semántico y RAG.
- **IE6:** documentación del mecanismo y limitaciones.

In [ ]:
"""
EP2 - IT Assist Multi-Agent Orchestration
DEMO de MEMORIA: Short-term vs Long-term en CrewAI

Proposito:
    1. Ejecutar el pipeline SIN memoria (baseline)
    2. Ejecutar el pipeline CON memoria (short-term + long-term)
    3. Comparar: con memoria, el agente recuerda contextos previos
       y produce respuestas más coherentes en consultas relacionadas.

Conceptos:
    - Short-term memory: contexto dentro de una ejecución (entre agentes).
      CrewAI lo maneja via `context` en Task y la memoria de conversacion.
    - Long-term memory: persistencia entre ejecuciones. CrewAI usa
      `memory=True` en el Agent para almacenar en una base local.
      Aquí lo simulamos con un JSON que persiste entre ejecuciones.

Uso:
    Ejecuta las celdas de este notebook para comparar memoria con y sin persistencia.
"""

import json
import os
import sys
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

load_dotenv()

from crewai import Agent, Task, Crew, Process
from crewai.tools import tool

from rag_tool import buscar_base_conocimiento


## 1. Memoria persistente

La memoria de largo plazo se almacena en `memoria_larga.json` y conserva como máximo 20 interacciones. Cada registro contiene timestamp, consulta y resumen.

La memoria de corto plazo se activa con `memory=True` en CrewAI. El archivo local est? excluido por `.gitignore`.

In [ ]:
# ─── Memoria persistente (long-term) ─────────────────────────────────────────

MEMORY_FILE = Path.cwd() / "memoria_larga.json"


def load_long_term_memory() -> list[dict]:
    """Carga la memoria a largo plazo (interacciones previas)."""
    if MEMORY_FILE.exists():
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return []


def save_long_term_memory(consulta: str, resumen: str):
    """Guarda una entrada en la memoria a largo plazo."""
    memories = load_long_term_memory()
    memories.append({
        "timestamp": datetime.now().isoformat(),
        "consulta": consulta,
        "resumen": resumen[:500],
    })
    # Mantener maximo 20 entradas
    memories = memories[-20:]
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(memories, f, ensure_ascii=False, indent=2)


def get_memory_context() -> str:
    """Retorna el contexto de memoria a largo plazo formateado."""
    memories = load_long_term_memory()
    if not memories:
        return "Sin interacciones previas almacenadas."
    parts = ["INTERACCIONES PREVIAS (memoria a largo plazo):"]
    for m in memories[-5:]:
        parts.append(f"  - [{m['timestamp'][:10]}] {m['consulta'][:80]}")
    return "\n".join(parts)


def _llm_kwargs() -> dict:
    return dict(
        model=os.getenv("LLM_MODEL", "llama3.2:3b"),
        api_key=os.getenv("LLM_API_KEY", "ollama"),
        base_url=os.getenv("LLM_BASE_URL", "http://localhost:11434/v1"),
        temperature=0.3,
    )


@tool("buscar_base_conocimiento_it")
def buscar_base_conocimiento_it(consulta: str) -> str:
    """Busca en la base de conocimiento de IT.

    Argumentos:
        consulta: La pregunta o tema a buscar.
    """
    return buscar_base_conocimiento(consulta)


## 2. Pipelines comparados

```text
Consulta → agente sin memoria → baseline
Consulta + contexto previo → agente con memoria → respuesta contextualizada
```

El contexto previo se incorpora al perfil del agente y a la descripción de la tarea. RAG aporta evidencia documental; la memoria aporta continuidad.

In [ ]:
# ─── Pipeline con y sin memoria ─────────────────────────────────────────────

def run_sin_memoria(consulta: str) -> str:
    """Ejecuta pipeline SIN memoria (baseline)."""
    ag = Agent(
        role="Asistente IT",
        goal="Responder consultas de IT con base en la base de conocimiento.",
        backstory="Asistente de mesa de ayuda. Busca en la base y responde.",
        tools=[buscar_base_conocimiento_it],
        llm=_llm_kwargs(),
        verbose=False,
        allow_delegation=False,
        memory=False,
    )
    t = Task(
        description=f"Consulta: {consulta}\nBusca y responde con fuentes.",
        expected_output="Respuesta con fuentes.",
        agent=ag,
    )
    crew = Crew(agents=[ag], tasks=[t], process=Process.sequential)
    result = crew.kickoff()
    return result.raw if result else "N/A"


def run_con_memoria(consulta: str) -> str:
    """Ejecuta pipeline CON memoria (short + long term)."""
    memory_ctx = get_memory_context()

    ag = Agent(
        role="Asistente IT con Memoria",
        goal="Responder consultas de IT usando memoria de interacciones previas y la base de conocimiento.",
        backstory=(
            "Asistente de mesa de ayuda CON MEMORIA. "
            "Recuerdas interacciones previas del usuario y las usas "
            "para dar contexto y consistencia en las respuestas. "
            + memory_ctx
        ),
        tools=[buscar_base_conocimiento_it],
        llm=_llm_kwargs(),
        verbose=False,
        allow_delegation=False,
        memory=True,  # Short-term memory activa en CrewAI
    )
    t = Task(
        description=(
            f"Consulta: {consulta}\n\n"
            f"CONTEXTO PREVIO: {memory_ctx}\n\n"
            "Usa la memoria de interacciones previas para dar contexto. "
            "Busca en la base de conocimiento y responde con fuentes. "
            "Si el usuario ya pregunto algo relacionado, referencia la interaccion previa."
        ),
        expected_output="Respuesta con fuentes y referencia a memoria previa.",
        agent=ag,
    )
    crew = Crew(agents=[ag], tasks=[t], process=Process.sequential)
    result = crew.kickoff()
    respuesta = result.raw if result else "N/A"

    # Guardar en memoria a largo plazo
    save_long_term_memory(consulta, respuesta)
    return respuesta


## 3. Ejecución, análisis y conclusiones

La demo procesa tres consultas y compara contenido, longitud, referencias previas y entradas persistidas. Una diferencia no garantiza mayor calidad: la memoria debe aportar contexto relevante sin inventar información.

### Limitaciones
- JSON sin búsqueda semántica sobre recuerdos.
- Ruta dependiente del directorio de ejecución.
- Resúmenes truncados.
- Falta una política de privacidad y eliminación para producción.

In [ ]:
# ─── Main: Demo comparativa ─────────────────────────────────────────────────

def main():
    consultas = [
        "Olvidé mi contraseña corporativa",
        "No puedo acceder a mi correo desde el trabajo",
        "Puedo instalar un antivirus personal en mi PC?",
    ]

    print("=" * 70)
    print("EP2 - IT Assist | DEMO MEMORIA (Short-term + Long-term)")
    print("=" * 70)

    for i, consulta in enumerate(consultas, 1):
        print(f"\n{'─' * 70}")
        print(f"CONSULTA {i}: {consulta}")
        print(f"{'─' * 70}")

        print("\n[1] SIN MEMORIA (baseline):")
        r1 = run_sin_memoria(consulta)
        print(f"    {r1[:300]}..." if len(r1) > 300 else f"    {r1}")

        print("\n[2] CON MEMORIA (short + long term):")
        r2 = run_con_memoria(consulta)
        print(f"    {r2[:300]}..." if len(r2) > 300 else f"    {r2}")

        # Diferencia
        print(f"\n[COMPARACION]")
        if r1.strip() == r2.strip():
            print("    -> Respuestas identicas (no hubo diferencia observable)")
        else:
            print("    -> Respuestas DIFIERENTES (la memoria influyo en el output)")
            print(f"    Sin memoria: {len(r1)} chars")
            print(f"    Con memoria: {len(r2)} chars")

    print(f"\n{'=' * 70}")
    print("MEMORIA A LARGO PLAZO (archivo persistido):")
    print(f"{'=' * 70}")
    for m in load_long_term_memory():
        print(f"  [{m['timestamp'][:16]}] {m['consulta'][:60]}")
    print(f"\nTotal entradas: {len(load_long_term_memory())}")
    print(f"Archivo: {MEMORY_FILE}")


if __name__ == "__main__":
    main()
